# EDA - MIDV2020 Transforms

- Purpose: verify that each geometric (image, corners) transform in `src/data/transforms.py`
  moves the corner coordinates consistently with the actual image transform.
- Method: apply each transform to 20 randomly shuffled MIDV2020 samples and overlay the
  transformed corners on the transformed image. If the transform is correct, the polygon
  should line up exactly with the document edges.
- Corner order: TL, TR, BR, BL, normalized [0, 1].

In [22]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [23]:
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
from src.data.dataset import Dataset
from src.core.factory import get_transform
from src.data.transforms import (
    RandomHorizontalFlip, RandomVerticalFlip, RandomRotation,
    RandomAffine, RandomPerspective, RandomScale,
)
from src.utils.plot import show_samples, load_raw_samples, apply_transform

In [25]:
CSV_PATH = os.path.join(PROJECT_ROOT, "data/midv2020/gt_corners.csv")
NUM_SAMPLES = 20

## Helpers

## 1. RandomHorizontalFlip

In [29]:
random.seed(0)
raw_samples = load_raw_samples(CSV_PATH, NUM_SAMPLES)
transform = RandomHorizontalFlip(p=1.0)
images, corners_list = apply_transform(transform, raw_samples)

In [ ]:
show_samples(images, corners_list, title="RandomHorizontalFlip (p=1.0)")

## 2. RandomVerticalFlip

In [31]:
random.seed(0)
raw_samples = load_raw_samples(CSV_PATH, NUM_SAMPLES)
transform = RandomVerticalFlip(p=1.0)
images, corners_list = apply_transform(transform, raw_samples)

In [ ]:
show_samples(images, corners_list, title="RandomVerticalFlip (p=1.0)")

## 3. RandomRotation

In [33]:
random.seed(0)
raw_samples = load_raw_samples(CSV_PATH, NUM_SAMPLES)
transform = RandomRotation(degrees=15)
images, corners_list = apply_transform(transform, raw_samples)

In [ ]:
show_samples(images, corners_list, title="RandomRotation (degrees=15)")

## 4. RandomAffine

In [35]:
random.seed(0)
raw_samples = load_raw_samples(CSV_PATH, NUM_SAMPLES)
transform = RandomAffine(degrees=15, translate=(0.1, 0.1), scale_range=(0.85, 1.15), shear=10)
images, corners_list = apply_transform(transform, raw_samples)

In [ ]:
show_samples(images, corners_list, title="RandomAffine (degrees=15, shear=10)")

## 5. RandomPerspective

In [37]:
random.seed(0)
raw_samples = load_raw_samples(CSV_PATH, NUM_SAMPLES)
transform = RandomPerspective(distortion_scale=0.15, p=1.0)
images, corners_list = apply_transform(transform, raw_samples)

In [ ]:
show_samples(images, corners_list, title="RandomPerspective (distortion_scale=0.15)")

## 6. RandomScale

In [39]:
random.seed(0)
raw_samples = load_raw_samples(CSV_PATH, NUM_SAMPLES)
transform = RandomScale(scale_range=(0.8, 1.2))
images, corners_list = apply_transform(transform, raw_samples)

In [ ]:
show_samples(images, corners_list, title="RandomScale (scale_range=(0.8, 1.2))")

## 7. Compose - actual train pipeline (get_transform)

Reproduces the geometric transforms that `get_transform(split="train")` actually applies
(`RandomHorizontalFlip`, `RandomVerticalFlip`, `RandomRotation` - `ColorJitter` and
`GaussianBlur` are image-only and do not affect corners).

In [41]:
random.seed(0)
raw_samples = load_raw_samples(CSV_PATH, NUM_SAMPLES)
pipeline = [RandomHorizontalFlip(p=0.5), RandomVerticalFlip(p=0.5), RandomRotation(degrees=5.0)]

images, corners_list = [], []
for image, corners in raw_samples:
    for t in pipeline:
        image, corners = t(image, corners)
    images.append(image)
    corners_list.append(corners)

In [ ]:
show_samples(images, corners_list, title="Compose - train geometric pipeline")